In [1]:
import pandas as pd
import numpy as np

In [2]:
df_urgency = pd.read_csv('../DataPreProcessing/mergeSet.csv', encoding='latin1')

In [3]:
df_urgency.columns

Index(['Unnamed: 0', 'subject', 'body', 'queue', 'priority', 'language'], dtype='object')

In [4]:
df_urgency.drop(columns=["Unnamed: 0"], inplace=True)


In [5]:
df_urgency.head()

,subject,body,queue,priority,language
0,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...",Technical Support,high,en
1,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Returns and Exchanges,medium,en
2,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",Billing and Payments,low,en
3,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Sales and Pre-Sales,medium,en
4,Feature Query,"Dear Customer Support,\n\nI hope this message ...",Technical Support,high,en


In [6]:
df_urgency['priority'].value_counts()

priority
medium    11570
high      10917
low        5774
Name: count, dtype: int64

In [7]:
df_urgency.drop(['language'], axis=1, inplace=True)

In [8]:
df_urgency.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28261 entries, 0 to 28260
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   24622 non-null  object
 1   body      28260 non-null  object
 2   queue     28261 non-null  object
 3   priority  28261 non-null  object
dtypes: object(4)
memory usage: 883.3+ KB


In [9]:
import re
import string

In [10]:
def clean_text(text):
    text = text.lower()
    text = text.replace("\n", " ").replace("\r", " ")
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [11]:
df_urg = df_urgency.copy()

In [12]:
df_urg.dropna(inplace=True)

In [13]:
df_urg["combined_text"] = df_urg["subject"] + " " + df_urg["body"]
df_urg["clean_text"] = df_urg["combined_text"].apply(clean_text)

In [14]:
df_urg[["combined_text", "clean_text"]].head(3)

,combined_text,clean_text
0,"Account Disruption Dear Customer Support Team,...",account disruption dear customer support teamn...
1,Query About Smart Home System Integration Feat...,query about smart home system integration feat...
2,Inquiry Regarding Invoice Details Dear Custome...,inquiry regarding invoice details dear custome...


In [15]:
# Remove newline characters
text = df_urg.replace("\n", " ").replace("\r", " ")

In [16]:
from sklearn.preprocessing import LabelEncoder
le_queue = LabelEncoder()
df_urg["encoded_queue"] = le_queue.fit_transform(df_urg["queue"])

In [17]:
urgent_keywords = [
    "urgent", "asap", "immediately", "right away",
    "critical", "emergency", "priority",
    "important", "high priority",

    # System failure
    "server down", "system down", "outage",
    "not working", "failure", "failed",
    "error", "crash", "blocked",
    "cannot access", "can't access",
    "unable to access", "not responding",

    # Escalation tone
    "escalate", "escalation",
    "complaint", "serious issue",
    "major issue", "severe",
    "breach", "security issue",

    # Time pressure
    "today", "now", "immediate action",
    "within hours", "deadline",
    "by end of day", "eod",
    "as soon as possible",

    # Payment/financial risk
    "payment failed", "transaction failed",
    "refund pending", "money deducted"
]

In [18]:
medium_keywords = [
    "please check",
    "follow up",
    "update",
    "status",
    "waiting",
    "pending",
    "reminder",
    "kindly respond",
    "request",
    "assistance needed",
    "help needed"
]

In [19]:
low_keywords = [
    "no rush",
    "whenever possible",
    "at your convenience",
    "just checking",
    "fyi",
    "for your information"
]

In [20]:
def urgency_signal(text):
    
    count = 0
    
    for word in urgent_keywords:
        if word in text:
            count += 3
            
    for word in medium_keywords:
        if word in text:
            count += 1
            
    for word in low_keywords:
        if word in text:
            count -= 2
            
    # Extra punctuation signals
    count += text.count("!") * 1
    
    return count

In [21]:
df_urg["urgency_signal"] = df_urg["clean_text"].apply(urgency_signal)

# Binary urgency presence
df_urg["has_urgency_word"] = (df_urg["urgency_signal"] > 0).astype(int)

In [22]:
df_urg["text_length"] = df_urg["clean_text"].apply(len)

In [23]:
df_urg["exclamation_count"] = df_urg["combined_text"].str.count("!")
df_urg["question_count"] = df_urg["combined_text"].str.count(r"\?")

In [24]:
from sklearn.utils import resample

df_high = df_urg[df_urg["priority"] == "high"]
df_medium = df_urg[df_urg["priority"] == "medium"]
df_low = df_urg[df_urg["priority"] == "low"]

df_low_upsampled = resample(
    df_low,
    replace=True,
    n_samples=8000,
    random_state=42
)

df_balanced = pd.concat([df_high, df_medium, df_low_upsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42)

print(df_balanced["priority"].value_counts())

priority
medium    10134
high       9454
low        8000
Name: count, dtype: int64


In [25]:
from sklearn.model_selection import train_test_split
X_text_raw = df_urg["clean_text"]
y = df_urg["priority"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text_raw,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [26]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [27]:
# Word features
tfidf_word = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,3),
    min_df=2,
    max_df=0.9
)

# Character features
tfidf_char = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3,5),
    max_features=10000
)

In [28]:
X_train_word = tfidf_word.fit_transform(X_train_text)
X_test_word = tfidf_word.transform(X_test_text)
X_train_char = tfidf_char.fit_transform(X_train_text)
X_test_char = tfidf_char.transform(X_test_text)

In [29]:
# Training features
X_train_queue = np.array(df_urg.loc[X_train_text.index, "encoded_queue"]).reshape(-1,1)
X_train_signal = np.array(df_urg.loc[X_train_text.index, "urgency_signal"]).reshape(-1,1)
X_train_binary = np.array(df_urg.loc[X_train_text.index, "has_urgency_word"]).reshape(-1,1)
X_train_length = np.array(df_urg.loc[X_train_text.index, "text_length"]).reshape(-1,1)

# Testing features
X_test_queue = np.array(df_urg.loc[X_test_text.index, "encoded_queue"]).reshape(-1,1)
X_test_signal = np.array(df_urg.loc[X_test_text.index, "urgency_signal"]).reshape(-1,1)
X_test_binary = np.array(df_urg.loc[X_test_text.index, "has_urgency_word"]).reshape(-1,1)
X_test_length = np.array(df_urg.loc[X_test_text.index, "text_length"]).reshape(-1,1)

In [30]:
from scipy.sparse import hstack

X_train_text_vec = hstack([X_train_word, X_train_char])
X_test_text_vec = hstack([X_test_word, X_test_char])

In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler(with_mean=False)

numeric_train = np.hstack([
    X_train_queue,
    X_train_signal,
    X_train_binary,
    X_train_length
])

numeric_test = np.hstack([
    X_test_queue,
    X_test_signal,
    X_test_binary,
    X_test_length
])

numeric_train_scaled = scaler.fit_transform(numeric_train)
numeric_test_scaled = scaler.transform(numeric_test)

In [32]:
X_train_final = hstack([X_train_text_vec, numeric_train_scaled])
X_test_final = hstack([X_test_text_vec, numeric_test_scaled])

In [33]:
from sklearn.svm import LinearSVC

urg_model = LinearSVC(
    class_weight="balanced",
    C=2.5   # try 0.5, 1.0, 1.5, 2.0
)

urg_model.fit(X_train_final, y_train)

y_pred = urg_model.predict(X_test_final)

c:\Users\MSI\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


In [34]:
from sklearn.metrics import classification_report, accuracy_score

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7116751269035533
              precision    recall  f1-score   support

        high       0.74      0.75      0.75      1891
         low       0.63      0.64      0.64      1007
      medium       0.72      0.71      0.72      2027

    accuracy                           0.71      4925
   macro avg       0.70      0.70      0.70      4925
weighted avg       0.71      0.71      0.71      4925



In [38]:
import pickle

# Save trained urgency model
with open("urgency_model.pkl", "wb") as f:
    pickle.dump(urg_model, f)

# Save word tfidf
with open("urgency_tfidf_word.pkl", "wb") as f:
    pickle.dump(tfidf_word, f)

# Save char tfidf
with open("urgency_tfidf_char.pkl", "wb") as f:
    pickle.dump(tfidf_char, f)

# Save scaler for numeric features
with open("urgency_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print("Urgency model + vectorizers + scaler saved successfully!")

Urgency model + vectorizers + scaler saved successfully!
